# CRANE Remote Ollama on Kaggle
Runs Ollama + exposes it via Cloudflare Tunnel. Paste the tunnel URL into opencode.jsonc as a provider.

**Runtime:** GPU T4 x2 recommended (Settings → Accelerator → GPU T4 x2)

**Run all cells top to bottom. Cell 5 prints your URL.**

In [ ]:
# Cell 1 — Install Ollama (needs zstd + pciutils on Kaggle's image)
import subprocess, os
os.environ['OLLAMA_HOST'] = '0.0.0.0'
os.environ['OLLAMA_MODELS'] = '/kaggle/working/ollama_models'
os.makedirs('/kaggle/working/ollama_models', exist_ok=True)

subprocess.run('apt-get update -qq && apt-get install -y -qq zstd pciutils', shell=True, check=True)
result = subprocess.run(
    'curl -fsSL https://ollama.com/install.sh | sh',
    shell=True, capture_output=True, text=True
)
print(result.stdout[-500:])
assert os.path.exists('/usr/local/bin/ollama'), 'Ollama install FAILED:\n' + result.stdout[-800:] + result.stderr[-800:]
print('Ollama installed')

In [ ]:
# Cell 2 — Start Ollama server
import subprocess, time

env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'
env['OLLAMA_MODELS'] = '/kaggle/working/ollama_models'

proc = subprocess.Popen(
    ['/usr/local/bin/ollama', 'serve'],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(4)

import urllib.request
try:
    urllib.request.urlopen('http://localhost:11434').read()
    print('Ollama is UP on port 11434')
except:
    print('ERROR: Ollama not responding')

In [ ]:
# Cell 3 — Pull uncensored lead model (dolphin3:8b)
# Change MODEL to any Ollama model you want
MODEL = 'dolphin3:8b'

print(f'Pulling {MODEL} — this takes 5-15 min...')
result = subprocess.run(
    ['/usr/local/bin/ollama', 'pull', MODEL],
    env=env, capture_output=True, text=True
)
print(result.stdout[-300:] or result.stderr[-300:])
print(f'{MODEL} ready' if result.returncode == 0 else 'PULL FAILED')

In [ ]:
# Cell 4 — Install and start Cloudflare Tunnel
import subprocess, time

subprocess.run(
    'curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
    '-o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared',
    shell=True, check=True
)
print('cloudflared installed')

In [ ]:
# Cell 5 — Start tunnel and print your URL
# THIS IS THE URL YOU COPY INTO opencode.jsonc
import subprocess, threading, time, re

tunnel_url = [None]

def run_tunnel():
    proc = subprocess.Popen(
        ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
        stderr=subprocess.PIPE, stdout=subprocess.DEVNULL, text=True
    )
    for line in proc.stderr:
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if m:
            tunnel_url[0] = m.group()
            break

t = threading.Thread(target=run_tunnel, daemon=True)
t.start()

for _ in range(30):
    if tunnel_url[0]: break
    time.sleep(1)

url = tunnel_url[0]
print('=' * 60)
print(f'TUNNEL URL: {url}')
print('=' * 60)
print()
print('Add this to ~/.config/opencode/opencode.jsonc:')
print(f'''"kaggle": {{
  "npm": "@ai-sdk/openai-compatible",
  "name": "Kaggle GPU",
  "options": {{
    "baseURL": "{url}/v1",
    "apiKey": "EMPTY"
  }},
  "models": {{
    "dolphin3:8b": {{"name": "Dolphin 3 8B (Kaggle)"}}
  }}
}}''')
print()
print('Keep this notebook running to maintain the tunnel.')

In [ ]:
# Cell 6 — Keep-alive (run last, keeps session active)
import time
print('Session alive. Ctrl-C or interrupt kernel to stop.')
while True:
    time.sleep(60)
    print('.', end='', flush=True)